# 06 — Optional Real Azure ADLS External Location

This notebook is **Run-All safe**.

Keep `enable_external_adls=false` in Free Edition.

Set it to true only when you have:
- ADLS Gen2 account/container,
- Unity Catalog storage credential,
- required privileges.


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / marker).exists():
        PROJECT_ROOT = str(candidate)
        break

if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
dbutils.widgets.dropdown(
    "enable_external_adls", "false", ["false", "true"],
    "Enable real ADLS"
)
dbutils.widgets.text("storage_account", "", "Storage Account")
dbutils.widgets.text("container", "", "Container")
dbutils.widgets.text("storage_credential", "", "Storage Credential")
dbutils.widgets.text(
    "external_location_name",
    "retail_training_ext_loc",
    "External Location Name"
)

enable_external_adls = (
    dbutils.widgets.get("enable_external_adls").lower() == "true"
)

storage_account = dbutils.widgets.get("storage_account").strip()
container = dbutils.widgets.get("container").strip()
storage_credential = dbutils.widgets.get("storage_credential").strip()
external_location_name = dbutils.widgets.get("external_location_name").strip()


In [0]:
from storage import abfss_uri

if not enable_external_adls:
    print("SKIPPED: real ADLS mode is disabled.")
else:
    if not all([
        storage_account, container,
        storage_credential, external_location_name
    ]):
        raise ValueError(
            "Fill all ADLS parameters when enable_external_adls=true."
        )

    container_url = abfss_uri(
        container=container,
        storage_account=storage_account,
    )
    print("ADLS URL:", container_url)


In [0]:
if enable_external_adls:
    safe_location = external_location_name.replace("`", "``")
    safe_credential = storage_credential.replace("`", "``")

    sql_text = (
        f"CREATE EXTERNAL LOCATION IF NOT EXISTS `{safe_location}` "
        f"URL '{container_url}' "
        f"WITH (STORAGE CREDENTIAL `{safe_credential}`) "
        f"COMMENT 'Retail Classes 27-29 training location'"
    )

    try:
        spark.sql(sql_text)
        print("External location created/verified.")
        display(
            spark.sql(
                f"DESCRIBE EXTERNAL LOCATION `{safe_location}`"
            )
        )
    except Exception as exc:
        print(
            "External-location setup could not complete. "
            "Check Azure storage, credential, and privileges."
        )
        print("Reason:", str(exc)[:1000])
else:
    print("No external-location DDL executed.")
